# ForkWise Live Demo Notebook

This notebook is the short, presentation-friendly runner for the **already deployed** ForkWise system.

It is designed for the situation where:
- Terraform is already done
- Ansible is already done
- pods are already up
- you want a clean sequence to demonstrate the system

## What this notebook covers

1. cluster and service health
2. live inference
3. feedback capture
4. request/feedback evidence in logs
5. retraining trigger path
6. training and evaluation logs
7. model version refresh check
8. serving metrics and monitoring stack checks
9. safeguarding code references to mention in the demo

## What to show outside the notebook

This notebook does **not** replace:
- the Mealie browser flow
- the object-store browser view
- the actual Grafana dashboards

Use this notebook as the backbone, then switch to those views at the right moments.

## Before Running

Open these in separate terminals and keep them running.

Terminal A:

```bash
ssh -N -L 8000:127.0.0.1:30080 -L 9000:127.0.0.1:30090 cc@<NODE1_FLOATING_IP>
```

Terminal B:

```bash
ssh -L 8001:127.0.0.1:8001 cc@<NODE1_FLOATING_IP> "kubectl port-forward svc/subst-feedback -n forkwise-data 8001:8001"
```

If Grafana is not exposed publicly, optionally open a third tunnel:

```bash
ssh -L 3000:127.0.0.1:3000 cc@<NODE1_FLOATING_IP> "kubectl port-forward svc/grafana -n monitoring-proj01 3000:3000"
```


In [ ]:
import json
import subprocess
import time
from pathlib import Path

import requests

NODE_IP = "REPLACE_WITH_NODE1_FLOATING_IP"
SERVING_URL = "http://localhost:8000"
FEEDBACK_URL = "http://localhost:8001"
REPO_ROOT = Path.home() / "ml-sys-ops-project"
SAMPLE_INPUT = REPO_ROOT / "serving/sample_data/input_sample.json"

def run(cmd, timeout=180, check=True):
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

def sshrun(cmd, timeout=180, check=True):
    return run(f'ssh -i /work/.ssh/id_rsa cc@{NODE_IP} "{cmd}"', timeout=timeout, check=check)

def pretty(obj):
    print(json.dumps(obj, indent=2))


## 1. Cluster Sanity Check

What to say:
- This confirms the canonical Chameleon and k3s deployment is up.
- We have a 3-node cluster and the app namespaces are present.


In [ ]:
sshrun("kubectl get nodes")
sshrun("kubectl get pods -n forkwise-app")
sshrun("kubectl get pods -n forkwise-serving")
sshrun("kubectl get pods -n forkwise-data")
sshrun("kubectl get pods -n monitoring-proj01", check=False)


## 2. Serving and Feedback Health

What to say:
- The serving API is alive.
- The feedback API is alive.
- This is the minimum proof that the online loop is reachable.


In [ ]:
serving_health = requests.get(f"{SERVING_URL}/health", timeout=10)
serving_health.raise_for_status()
pretty(serving_health.json())

feedback_health = requests.get(f"{FEEDBACK_URL}/health", timeout=10)
feedback_health.raise_for_status()
pretty(feedback_health.json())


## 3. Run One Live Prediction

What to say:
- This is the live ML feature backend.
- The response includes `request_id`, `model_version`, and `serving_version`.


In [ ]:
with open(SAMPLE_INPUT, "r", encoding="utf-8") as f:
    predict_payload = json.load(f)

predict_response = requests.post(
    f"{SERVING_URL}/predict",
    json=predict_payload,
    timeout=20,
)
predict_response.raise_for_status()
predict_json = predict_response.json()
pretty(predict_json)

request_id = predict_json["request_id"]
model_version_before = predict_json.get("model_version")
print("request_id:", request_id)
print("model_version_before:", model_version_before)


## 4. Capture Feedback

What to say:
- This shows how user outcomes are fed back into the system.
- The same `request_id` closes the loop between prediction and feedback.


In [ ]:
feedback_payload = {
    "request_id": request_id,
    "recipe_id": str(predict_json["recipe_id"]),
    "missing_ingredient": predict_json["missing_ingredient"],
    "suggested_substitution": predict_json["substitutions"][0]["ingredient"],
    "user_accepted": True,
    "model_version": model_version_before,
}

feedback_response = requests.post(
    f"{FEEDBACK_URL}/feedback",
    json=feedback_payload,
    timeout=20,
)
feedback_response.raise_for_status()
pretty(feedback_response.json())


## 5. Show Request and Feedback Evidence

What to say:
- Serving logs online requests.
- The feedback service logs user responses.
- Outside the notebook, this is also the right moment to open the object-store browser and show `logs/requests/` and `logs/feedback/`.


In [ ]:
sshrun("kubectl logs deployment/substitution-serving -n forkwise-serving --tail=50", timeout=60, check=False)
sshrun("kubectl logs deployment/subst-feedback -n forkwise-data --tail=50", timeout=60, check=False)


## 6. Trigger Batch Processing

What to say:
- Batch processing turns logged feedback into retraining-ready data.
- This is where processed datasets and retraining triggers are created.
- Outside the notebook, this is also a good point to show object storage under `data/processed/` and `data/triggers/`.


In [ ]:
sshrun("kubectl patch configmap forkwise-data-config -n forkwise-data --type merge -p '{\"data\":{\"MIN_NEW_SAMPLES\":\"1\"}}'", timeout=60, check=False)
sshrun("kubectl delete job batch-smoke -n forkwise-data --ignore-not-found=true", timeout=60, check=False)
sshrun("kubectl create job batch-smoke --from=cronjob/batch-pipeline -n forkwise-data", timeout=60)
sshrun("kubectl wait --for=condition=complete job/batch-smoke -n forkwise-data --timeout=240s", timeout=260, check=False)
sshrun("kubectl logs -n forkwise-data job/batch-smoke --tail=200", timeout=60, check=False)


## 7. Trigger Training and Evaluation

What to say:
- The training job consumes the trigger.
- Evaluation happens before publish.
- This is the right way to show the training role in the demo.
- Do not force MLflow here unless it is already live and populated.


In [ ]:
sshrun("kubectl delete job training-smoke -n forkwise-data --ignore-not-found=true", timeout=60, check=False)
sshrun("kubectl create job training-smoke --from=cronjob/training-trigger -n forkwise-data", timeout=60)
sshrun("kubectl wait --for=condition=complete job/training-smoke -n forkwise-data --timeout=420s", timeout=440, check=False)
sshrun("kubectl logs -n forkwise-data job/training-smoke --tail=250", timeout=60, check=False)


## 8. Verify Serving Model Version Again

What to say:
- This checks whether serving picked up the new model artifacts.
- Outside the notebook, this is also a good time to show object storage under `models/production/`.


In [ ]:
health_after = requests.get(f"{SERVING_URL}/health", timeout=10)
health_after.raise_for_status()
pretty(health_after.json())
print("model_version_before:", model_version_before)
print("model_version_after:", health_after.json().get("model_version"))


## 9. Serving Metrics and Monitoring Stack

What to say:
- Serving exports Prometheus metrics.
- The platform monitoring stack exists.
- After this cell, switch to Grafana in the browser and show the real dashboards.


In [ ]:
metrics_text = requests.get(f"{SERVING_URL}/metrics", timeout=10).text
interesting = [line for line in metrics_text.splitlines() if line.startswith("subst_")]
print("\n".join(interesting[:120]))

sshrun("kubectl get pods -n monitoring-proj01", timeout=60, check=False)
sshrun("kubectl get svc -n monitoring-proj01", timeout=60, check=False)


## 10. Optional Drift Job

What to say:
- This demonstrates data-quality and drift monitoring.
- It supports the data role's monitoring/evaluation story.


In [ ]:
sshrun("kubectl delete job drift-smoke -n forkwise-data --ignore-not-found=true", timeout=60, check=False)
sshrun("kubectl create job drift-smoke --from=cronjob/drift-monitor -n forkwise-data", timeout=60, check=False)
sshrun("kubectl wait --for=condition=complete job/drift-smoke -n forkwise-data --timeout=240s", timeout=260, check=False)
sshrun("kubectl logs -n forkwise-data job/drift-smoke --tail=200", timeout=60, check=False)


## 11. Safeguarding References

Open these files in the editor while narrating the safeguarding plan:

- `data/feedback_endpoint.py`
- `data/batch_pipeline.py`
- `data/drift_monitor.py`
- `training/evaluate.py`
- `serving/fastapi_onnx/serve_onnx.py`
- `serving/scripts/check_rollback.py`
- `infra/k8s/platform/prometheus-configmap.yaml`
- `infra/k8s/platform/grafana-dashboard-configmap.yaml`

Suggested narration:
- privacy: feedback does not require user identity
- transparency: responses include model and serving version metadata
- accountability: request logs, feedback logs, and rollback paths exist
- robustness: evaluation gate, rollout monitoring, and rollback are present
- fairness/evaluation: training evaluation is explicit and reviewable


In [ ]:
sshrun("kubectl patch configmap forkwise-data-config -n forkwise-data --type merge -p '{\"data\":{\"MIN_NEW_SAMPLES\":\"50\"}}'", timeout=60, check=False)
print("Demo cleanup complete.")
